# Mapping migration


## Set up

To get started on this notebook, you’ll need to restore any variables from previous notebooks to your workspace. To save time and memory, make sure to specify which variables you want to load.

In [1]:
%store -r

You will also need to import any libraries you are using in this notebook. In this notebook, you will need some additional packages to make interactive maps with vector data and access pre-defined month names.

In [2]:
### import packages

### libraries for dynamic mapping
### goal: make a map with a slider that changes the map as the month changes

### for coordinate reference systems
import cartopy.crs as ccrs 

### customize dynamic map
import panel as pn

### dynamic plot with geodataframe
import hvplot.pandas
import holoviews as hv

### get month names (optional activity at the end)
import calendar

### file management
import os
import pathlib 

## Step 5: Make an interactive map of GBIF observations

### Step 5A: Create a simplified `GeoDataFrame` for plotting

Plotting larger files can be time consuming. The code below will streamline plotting with `hvplot` by simplifying the geometry, projecting it to a Mercator projection that is compatible with `geoviews`, and cropping off areas in the Arctic:

1. Simplify the ecoregions with `.simplify(.05)`, and save it back to the `geometry` column.
2. Change the Coordinate Reference System (CRS) to Mercator with `.to_crs(ccrs.Mercator())`.
3. Use the plotting code that is already in the cell to check that the plotting runs quickly (less than a minute) and looks the way you want, making sure to change `gdf` to your `GeoDataFrame` name.

In [3]:
### plotting with just hvplot would be slow because complex geometries

### simplify the geometry to speed up processing
ecoreg_gdf.geometry = ecoreg_gdf.simplify(

    ### can increase the number above 0.05 to 0.1 to speed up
    .05, 

    ### set this to false to make it process faster
    preserve_topology=False)

### change the CRS to Mercator for mapping
ecoreg_gdf = ecoreg_gdf.to_crs(ccrs.Mercator())

### check that the plot runs in a reasonable amount of time

ecoreg_gdf.hvplot(geo = True, crs = ccrs.Mercator())

c:\Users\kjsie\anaconda3\envs\earth-analytics-python\Lib\site-packages\dask\dataframe\__init__.py:49: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


:Polygons   [Longitude,Latitude]

### Step 5B: Map migration over time

1. If applicable, replace any variable names with the names you defined previously.
2. Replace `column_name_used_for_ecoregion_color` and `column_name_used_for_slider` with the column names you wish to use.
3. Customize your plot with your choice of title, tile source, color map, and size.

Your plot will probably still change months very slowly in your Jupyter notebook, because it calculates each month’s plot as needed. Open up the saved HTML file to see faster performance!

In [4]:
### rename ecoregID to match occurrence_df
ecoreg_gdf = ecoreg_gdf.rename(columns = {'OBJECTID': 'ecoregion'})

### make ecoreg ID the index for merge with occurrence_df
ecoreg_gdf = ecoreg_gdf.set_index('ecoregion')
ecoreg_gdf.head

### join the occurrences with the plotting GeoDataFrame
occurrence_gdf = ecoreg_gdf.join(occurrence_df)

In [5]:
### activate panel and hvplot extensions
pn.extension()
hv.extension("bokeh")

Make the map with a basemap for the portfolio assignment

In [7]:
### get the plot bounds
xmin, ymin, xmax, ymax = occurrence_gdf.total_bounds

### define the slider widget
slider = pn.widgets.DiscreteSlider(
    name = 'month', 
    options = {calendar.month_name[i]: i for i in range(1, 13)}
)

### make the basemap tile object
carto_tiles = hv.Tiles(
    ("https://basemaps.cartocdn.com/rastertiles/voyager/{Z}/{X}/{Y}.png"
     "?key=cb1_446c_1_8946b413de0ace50f732dbf1"),
     name = "CARTO"
)

### plot the occurrence by month and ecoregion
migration_plot = (
    occurrence_gdf
    .hvplot(

        ### color by number of occurrences
        c = 'norm_occurrences',

        ### group by month
        groupby = 'month',

        ### set up for basemap tiles
        geo = True, crs = ccrs.Mercator(), 
        
        # tiles = carto_tiles,
        tiles = carto_tiles,

        ### add a title
        title = "Interactive map of Veery thrush migration",

        ### set plot boundaries
        xlim = (xmin, xmax), ylim = (ymin, ymax),

        ### set height and width of map
        frame_height = 600,
        frame_width = 900,

        ### put the slider widget at the bottom of the map
        widget_location = 'bottom',

        ### if you want to see the legend, change this to true
        ### note that the values for the different colors vary month to month
        colorbar = True,

        ### label slider widget
        widgets = {'month': slider}
    )
)


## wrap plot with panel
interactive_migration_plot = pn.panel(migration_plot)

interactive_migration_plot

BokehModel(combine_events=True, render_bundle={'docs_json': {'bd2fa895-bfea-43b1-a0a2-b0af13578e9c': {'version…

In [ ]:
### make path for the map
map_path_carto = os.path.join(data_dir, "migration_carto_demo.html")
map_path_carto

### export your map
migration_plot.save(
    map_path_carto,
    embed = True,
    resources = "inline",
)